# Live Data Walkthrough — Multi-Agent Finance Debate System

This notebook traces exactly how a real stock ticker becomes a debate scenario, step by step.

**Flow:**
```
User types "CRM"
  → Step 1: Look up CIK from SEC EDGAR ticker registry
  → Step 2: Fetch full GAAP facts for that CIK
  → Step 3: Extract quarterly financials (revenue, margins, FCF, …)
  → Step 4: Build the scenario dict the agents consume
  → Step 5: Agents receive the scenario and debate
```

In [1]:
import sys
sys.path.insert(0, ".")   # make sure project root is on path

import asyncio, json, httpx
from data_loader import (
    EDGAR_TICKERS_URL, EDGAR_HEADERS, EDGAR_BASE,
    TICKER_TO_CIK,
    lookup_cik_by_ticker,
    fetch_edgar_facts,
    _extract_quarterly,
    build_edgar_scenario,
    fetch_real_scenario,
    SCENARIOS,
)

TICKER = "MSFT"   # ← change this to any US public ticker
print("Ready. Ticker:", TICKER)

Ready. Ticker: MSFT


In [2]:
import importlib, data_loader
importlib.reload(data_loader)

from data_loader import (
    EDGAR_TICKERS_URL, EDGAR_HEADERS, EDGAR_BASE,
    TICKER_TO_CIK,
    lookup_cik_by_ticker,
    fetch_edgar_facts,
    _extract_quarterly,
    build_edgar_scenario,
    fetch_real_scenario,
    SCENARIOS,
)
print("data_loader reloaded — _extract_quarterly version is fresh")

data_loader reloaded — _extract_quarterly version is fresh


## Step 1 — CIK Lookup

Every SEC filing is indexed by a **CIK** (Central Index Key), not a ticker.  
`lookup_cik_by_ticker()` first checks the hardcoded `TICKER_TO_CIK` map; if not found it fetches `https://www.sec.gov/files/company_tickers.json` — a ~2 MB registry of all ~12 000 US public companies.

In [3]:
# Hardcoded fast-path (5 known tickers skip the network call)
print("Hardcoded CIK map:", TICKER_TO_CIK)

# Full lookup (uses network if ticker not in map above)
cik, company_name = await lookup_cik_by_ticker(TICKER)
print(f"\nTicker : {TICKER}")
print(f"CIK    : {cik}")
print(f"Name   : {company_name}")

Hardcoded CIK map: {'CRM': '0001108524', 'ADBE': '0000796343', 'NOW': '0001373715', 'WDAY': '0001327811', 'SNOW': '0001640147'}

Ticker : MSFT
CIK    : 0000789019
Name   : MICROSOFT CORP


## Step 2 — Fetch EDGAR Company Facts

With the CIK we call:
```
https://data.sec.gov/api/xbrl/companyfacts/CIK{cik}.json
```
This returns **every GAAP financial fact** the company has ever filed — income statement, balance sheet, cash flow — across all 10-Q and 10-K filings. No API key needed.

The JSON is large (~5–20 MB depending on company history). We only keep the `us-gaap` namespace.

In [4]:
facts = await fetch_edgar_facts(cik)

# Top-level keys in the response
print("Top-level keys :", list(facts.keys()))
print("Namespaces      :", list(facts["facts"].keys()))

us_gaap = facts["facts"]["us-gaap"]
print(f"\nus-gaap contains {len(us_gaap)} distinct financial concepts")
print("\nSample concept names:")
for name in list(us_gaap.keys())[:15]:
    print(" ", name)

Top-level keys : ['cik', 'entityName', 'facts']
Namespaces      : ['dei', 'us-gaap']

us-gaap contains 543 distinct financial concepts

Sample concept names:
  AccountsPayableCurrent
  AccountsReceivableNet
  AccountsReceivableNetCurrent
  AccountsReceivableNetNoncurrent
  AccruedIncomeTaxesCurrent
  AccruedIncomeTaxesNoncurrent
  AccumulatedDepreciationDepletionAndAmortizationPropertyPlantAndEquipment
  AccumulatedOtherComprehensiveIncomeLossAvailableForSaleSecuritiesAdjustmentNetOfTax
  AccumulatedOtherComprehensiveIncomeLossCumulativeChangesInNetGainLossFromCashFlowHedgesEffectNetOfTax
  AccumulatedOtherComprehensiveIncomeLossForeignCurrencyTranslationAdjustmentNetOfTax
  AccumulatedOtherComprehensiveIncomeLossNetOfTax
  AcquiredFiniteLivedIntangibleAssetAmount
  AcquiredFiniteLivedIntangibleAssetWeightedAverageUsefulLife
  AdvertisingExpense
  AllocatedShareBasedCompensationExpense


## Step 3 — Extract Quarterly Financials

Each GAAP concept looks like this:
```json
"Revenues": {
  "units": {
    "USD": [
      {"start": "2024-02-01", "end": "2024-04-30", "val": 9133000000,
       "form": "10-Q", "fp": "Q1", "filed": "2024-06-06", ...},
      ...
    ]
  }
}
```

`_extract_quarterly()` filters to **true quarterly periods** (60–105 days) and merges across **all candidate field names** before deduplicating by end date.

> **Why merge?** Companies that adopted ASC 606 (revenue recognition standard, ~2018) switched from `Revenues` to `RevenueFromContractWithCustomerExcludingAssessedTax`. Without merging, stopping at the first matching field would return only pre-2018 data for those companies.

In [5]:
# Revenue — tries multiple GAAP field names in order
revenue_data = _extract_quarterly(
    us_gaap,
    "Revenues",
    "RevenueFromContractWithCustomerExcludingAssessedTax",
    "SalesRevenueNet",
    "RevenueFromContractWithCustomerIncludingAssessedTax",
)

print(f"Found {len(revenue_data)} quarterly revenue entries\n")
print(f"{'End Date':<14} {'Period':<6} {'Revenue ($B)':>14}  {'Form'}")
print("-" * 44)
for q in revenue_data[:8]:
    print(f"{q['end']:<14} {q.get('fp','?'):<6} {q['val']/1e9:>13.2f}B  {q['form']}")

Found 53 quarterly revenue entries

End Date       Period   Revenue ($B)  Form
--------------------------------------------
2025-12-31     Q2             81.27B  10-Q
2025-09-30     Q1             77.67B  10-Q
2025-03-31     Q3             70.07B  10-Q
2024-12-31     Q2             69.63B  10-Q
2024-09-30     Q1             65.58B  10-Q
2024-03-31     Q3             61.86B  10-Q
2023-12-31     Q2             62.02B  10-Q
2023-09-30     Q1             56.52B  10-Q


In [8]:
# ── Diagnostic: find ALL revenue-related fields in this company's EDGAR data ──
from datetime import datetime as dt

revenue_fields = []
for field_name, field_data in us_gaap.items():
    if "revenue" not in field_name.lower() and "sales" not in field_name.lower():
        continue
    units_usd = field_data.get("units", {}).get("USD", [])
    # Check for quarterly 10-Q entries with ~90-day duration
    quarterly = []
    for u in units_usd:
        if u.get("form") != "10-Q":
            continue
        start, end = u.get("start", ""), u.get("end", "")
        if start and end:
            try:
                days = (dt.fromisoformat(end) - dt.fromisoformat(start)).days
                if 60 <= days <= 105:
                    quarterly.append(u)
            except ValueError:
                pass
    if quarterly:
        quarterly.sort(key=lambda x: x.get("end", ""), reverse=True)
        most_recent = quarterly[0]
        revenue_fields.append((most_recent["end"], field_name, most_recent["val"], len(quarterly)))

revenue_fields.sort(reverse=True)
print(f"{'Most Recent':<14} {'Field Name':<65} {'Value ($B)':>10}  {'#Qtrs'}")
print("-" * 100)
for end, name, val, count in revenue_fields:
    print(f"{end:<14} {name:<65} {val/1e9:>9.2f}B  {count}")

Most Recent    Field Name                                                        Value ($B)  #Qtrs
----------------------------------------------------------------------------------------------------
2025-12-31     RevenueFromContractWithCustomerExcludingAssessedTax                   81.27B  46
2025-12-31     ProceedsFromMaturitiesPrepaymentsAndCallsOfAvailableForSaleSecurities     12.42B  98
2025-12-31     OtherComprehensiveIncomeLossAvailableForSaleSecuritiesAdjustmentNetOfTax     -0.16B  82
2020-09-30     ContractWithCustomerLiabilityRevenueRecognized                        20.85B  3
2018-03-31     SalesRevenueNet                                                       26.82B  42
2018-03-31     SalesRevenueGoodsNet                                                  15.11B  12
2018-03-31     ProceedsFromSaleOfAvailableForSaleSecurities                          26.26B  52
2018-03-31     IncreaseDecreaseInDeferredRevenue                                      0.09B  42
2018-03-31     CostOfR

In [9]:
# Show all other financial fields we extract
fields = {
    "Gross Profit":      ("GrossProfit",),
    "Operating Expenses":("OperatingExpenses",),
    "Operating Income":  ("OperatingIncomeLoss",),
    "D&A":               ("DepreciationDepletionAndAmortization", "DepreciationAndAmortization"),
    "Cash from Ops":     ("NetCashProvidedByUsedInOperatingActivities",),
    "CapEx":             ("PaymentsToAcquirePropertyPlantAndEquipment",),
}

print(f"{'Field':<22} {'Most Recent Quarter ($M)':>26}  {'End Date'}")
print("-" * 60)
for label, names in fields.items():
    data = _extract_quarterly(us_gaap, *names)
    if data:
        q = data[0]
        print(f"{label:<22} {q['val']/1e6:>25.1f}M  {q['end']}")
    else:
        print(f"{label:<22} {'N/A — not in EDGAR filings':>26}")

Field                    Most Recent Quarter ($M)  End Date
------------------------------------------------------------
Gross Profit                             55295.0M  2025-12-31
Operating Expenses                       17020.0M  2025-12-31
Operating Income                         38275.0M  2025-12-31
D&A                    N/A — not in EDGAR filings
Cash from Ops                            35758.0M  2025-12-31
CapEx                                    29876.0M  2025-12-31


## Step 4 — Build the Scenario Dict

`build_edgar_scenario()` combines all the extracted fields into the dict structure the agents expect.  
Derived metrics:
- **YoY growth** = (latest quarter − same quarter 4 entries back) / prior × 100  
- **Gross margin** = Gross Profit / Revenue × 100  
- **EBITDA** = Operating Income + D&A  
- **FCF** = Cash from Ops − CapEx  
- **Forward guidance** = estimated as ±3% / +5% around the most recent quarter

In [10]:
scenario = build_edgar_scenario(TICKER, company_name, facts)

# Pretty-print the full scenario dict
print(json.dumps(scenario, indent=2, default=str))

{
  "company": "MICROSOFT CORP (NASDAQ/NYSE: MSFT)",
  "period": "Q2 2025",
  "next_period": "Q3 2025",
  "context": "Live SEC EDGAR data \u2014 most recent 10-Q (2025-12-31)",
  "data_source": "SEC EDGAR XBRL (10-Q filed 2025-12-31)",
  "financials": {
    "revenue_q3_actual": 81273000000,
    "revenue_q3_prior_year": 65585000000,
    "revenue_growth_yoy": 23.92,
    "gross_margin": 68.0,
    "operating_expenses": 17020000000,
    "ebitda": 38275000000,
    "ebitda_margin": 47.1,
    "free_cash_flow": 5882000000,
    "net_revenue_retention": null,
    "customer_churn_rate": null
  },
  "forward_guidance": {
    "q4_revenue_low": 78834810000,
    "q4_revenue_high": 85336650000,
    "q4_opex_projected": 17360400000,
    "fy2024_revenue_target": 333219300000
  },
  "macro_context": {
    "fed_funds_rate": 4.33,
    "data_note": "Forward guidance estimated from trailing quarter; macro from public sources"
  },
  "risks": [
    "Forward guidance estimated \u2014 check latest earnings call 

In [12]:
# Compare the derived metrics side by side
fin = scenario["financials"]
fwd = scenario["forward_guidance"]

print("=== FINANCIALS ===")
print(f"  Revenue (latest Q)     : ${fin['revenue_q3_actual']/1e9:.2f}B")
print(f"  Revenue (prior year Q) : ${fin['revenue_q3_prior_year']/1e9:.2f}B")
print(f"  YoY Growth             : {fin['revenue_growth_yoy']:.1f}%")
print(f"  Gross Margin           : {fin['gross_margin']}%")
print(f"  Operating Expenses     : ${fin['operating_expenses']/1e9:.2f}B")
print(f"  EBITDA                 : ${fin['ebitda']/1e9:.2f}B  ({fin['ebitda_margin']}% margin)")
print(f"  Free Cash Flow         : ${fin['free_cash_flow']/1e9:.2f}B")

print("\n=== FORWARD GUIDANCE (estimated) ===")
print(f"  Next Q range           : ${fwd['q4_revenue_low']/1e9:.2f}B – ${fwd['q4_revenue_high']/1e9:.2f}B")
print(f"  Next Q OpEx            : ${fwd['q4_opex_projected']/1e9:.2f}B")
print(f"  Full-Year target       : ${fwd['fy2024_revenue_target']/1e9:.2f}B")

print("\n=== QUALITATIVE (static placeholders) ===")
print("  Risks:")
for r in scenario["risks"]:
    print(f"    • {r}")
print("  Opportunities:")
for o in scenario["opportunities"]:
    print(f"    • {o}")

=== FINANCIALS ===
  Revenue (latest Q)     : $81.27B
  Revenue (prior year Q) : $65.58B
  YoY Growth             : 23.9%
  Gross Margin           : 68.0%
  Operating Expenses     : $17.02B
  EBITDA                 : $38.27B  (47.1% margin)
  Free Cash Flow         : $5.88B

=== FORWARD GUIDANCE (estimated) ===
  Next Q range           : $78.83B – $85.34B
  Next Q OpEx            : $17.36B
  Full-Year target       : $333.22B

=== QUALITATIVE (static placeholders) ===
  Risks:
    • Forward guidance estimated — check latest earnings call for official outlook
    • Macro headwinds: elevated interest rates and uncertain consumer demand
    • Competitive pricing pressure across the sector
  Opportunities:
    • AI integration opportunities across product lines
    • International expansion potential in underpenetrated markets
    • Operating leverage as revenue scales


## Step 5 — The Full Pipeline in One Call

`fetch_real_scenario(ticker)` chains all the above steps. It also short-circuits to the pre-built dict for `TCORP`, `MFGCO`, `REIT1`.

In [13]:
# Pre-built ticker → returns immediately from SCENARIOS dict (no network)
#prebuilt = await fetch_real_scenario("TCORP")
#print("TCORP source:", prebuilt["data_source"])

# Real ticker → runs all 3 network steps
live = await fetch_real_scenario(TICKER)
print(f"{TICKER} source  :", live["data_source"])

MSFT source  : SEC EDGAR XBRL (10-Q filed 2025-12-31)


## Step 6 — How Agents Consume the Scenario

Each agent node in `agents.py` receives `state["scenario"]` and reads specific fields to build its prompt. Here's what each agent actually sees.

In [14]:
s = live   # use the live EDGAR scenario
fin = s["financials"]
fwd = s["forward_guidance"]

period      = s.get("period", "Most Recent Quarter")
next_period = s.get("next_period", "Next Quarter")

print(f"Reporting period : {period}")
print(f"Guidance period  : {next_period}")

# ── Turn 1: Revenue Analyst prompt ──────────────────────────────────────────
ra_prompt = f"""Analyze the {period} results for {s['company']} and defend your {next_period} revenue guidance.

{period} ACTUALS:
- Revenue: ${fin['revenue_q3_actual']:,} ({fin['revenue_growth_yoy']:.1f}% YoY)
- Gross Margin: {fin['gross_margin']}%
- NRR: {fin.get('net_revenue_retention', 'N/A')}%
- Churn: {fin.get('customer_churn_rate', 'N/A')}%
- CAC Payback: {fin.get('cac_payback_months', 'N/A')} months

{next_period} GUIDANCE RANGE: ${fwd['q4_revenue_low']:,} – ${fwd['q4_revenue_high']:,}

KEY FACTS:
- {s['risks'][0]}
- {s['opportunities'][0]}
"""

print("\n=== REVENUE ANALYST PROMPT ===")
print(ra_prompt)

Reporting period : Q2 2025
Guidance period  : Q3 2025

=== REVENUE ANALYST PROMPT ===
Analyze the Q2 2025 results for MICROSOFT CORP (NASDAQ/NYSE: MSFT) and defend your Q3 2025 revenue guidance.

Q2 2025 ACTUALS:
- Revenue: $81,273,000,000 (23.9% YoY)
- Gross Margin: 68.0%
- NRR: None%
- Churn: None%
- CAC Payback: N/A months

Q3 2025 GUIDANCE RANGE: $78,834,810,000 – $85,336,650,000

KEY FACTS:
- Forward guidance estimated — check latest earnings call for official outlook
- AI integration opportunities across product lines



In [15]:
# ── Turn 2: Cost Analyst prompt (receives Turn 1 output) ────────────────────
ra_output_placeholder = "[Revenue Analyst's response from Turn 1 goes here]"

ca_prompt = f"""Cross-examine the Revenue Analyst's {next_period} projections for {s['company']}.

REVENUE ANALYST'S POSITION:
{ra_output_placeholder}

YOUR COST DATA:
- {period} OpEx: ${fin['operating_expenses']:,}
- {next_period} Projected OpEx: ${fwd['q4_opex_projected']:,}
- IT Budget Freeze affecting: {s['macro_context'].get('customer_segment_IT_budget_freeze_pct', 'N/A')}% of customer segment

STRUCTURAL RISKS:
{chr(10).join('- ' + r for r in s['risks'])}
"""

print("=== COST ANALYST PROMPT ===")
print(ca_prompt)

=== COST ANALYST PROMPT ===
Cross-examine the Revenue Analyst's Q3 2025 projections for MICROSOFT CORP (NASDAQ/NYSE: MSFT).

REVENUE ANALYST'S POSITION:
[Revenue Analyst's response from Turn 1 goes here]

YOUR COST DATA:
- Q2 2025 OpEx: $17,020,000,000
- Q3 2025 Projected OpEx: $17,360,400,000
- IT Budget Freeze affecting: N/A% of customer segment

STRUCTURAL RISKS:
- Forward guidance estimated — check latest earnings call for official outlook
- Macro headwinds: elevated interest rates and uncertain consumer demand
- Competitive pricing pressure across the sector



## Step 7 — Run the Full Debate (optional, costs ~$0.008)

Uncomment the cell below to run all 5 agent turns end-to-end with the live EDGAR data.

In [53]:
import os
# os.environ["ANTHROPIC_API_KEY"] = "sk-ant-..."   # set if not already in environment

# from agents import run_debate
#
# result = await run_debate(scenario=live)
#
# print("\n" + "="*60)
# print("CFO RECOMMENDATION")
# print("="*60)
# print(result["final_recommendation"])
# print()
# print(f"Escalation triggered : {result['escalation_triggered']}")
# print(f"Disagreement score   : {result['disagreement_score']:.0%}")
print("Uncomment the lines above to run the full debate.")

Uncomment the lines above to run the full debate.


## Summary — Data Flow

```
fetch_real_scenario("CRM")
│
├─ 1. lookup_cik_by_ticker("CRM")
│      checks TICKER_TO_CIK → "0001108524"
│      (fallback: GET sec.gov/files/company_tickers.json)
│
├─ 2. fetch_edgar_facts("0001108524")
│      GET data.sec.gov/api/xbrl/companyfacts/CIK0001108524.json
│      → ~10 MB JSON with every GAAP fact ever filed
│
├─ 3. build_edgar_scenario("CRM", "Salesforce Inc", facts)
│      _extract_quarterly(us_gaap, "Revenues", ...)       → revenue_data
│      _extract_quarterly(us_gaap, "GrossProfit")         → gross profit
│      _extract_quarterly(us_gaap, "OperatingIncomeLoss") → op income
│      _extract_quarterly(us_gaap, "DepreciationD&A")     → D&A
│      _extract_quarterly(us_gaap, "NetCashFromOps")      → CFO
│      _extract_quarterly(us_gaap, "CapEx")               → capex
│
│      Derives: YoY growth, gross margin %, EBITDA, FCF
│      Estimates: forward guidance = ±3%/+5% of latest quarter
│      Static: risks, opportunities (generic placeholders)
│
└─ returns scenario dict → agents.py FinanceState["scenario"]
```

**What EDGAR provides (real):** revenue, margins, EBITDA, FCF  
**What is estimated:** forward guidance range  
**What is static:** risks, opportunities, macro context